<a href="https://colab.research.google.com/github/gt-cse-6040/bootcamp/blob/main/SQL/syllabus/SQL3nb2_SQL_SUBQUERIES_FA25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Subqueries

In [4]:
!wget https://github.com/gt-cse-6040/bootcamp/raw/main/SQL/syllabus/NYC-311-2M_small.db

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
# create a connection to the database
import sqlite3 as db
import pandas as pd

# Connect to a database (or create one if it doesn't exist)
conn_nyc = db.connect('NYC-311-2M_small.db')

## Subqueries quick review:
- A subquery is a query nested within another SQL query, typically within the SELECT, WHERE, or FROM clause.

- Subqueries allow you to perform calculations or filtering within the context of the outer query.

- They are particularly useful when you need to filter results based on aggregated values like maximum, minimum, and average or are pivoting the data from rows to columns like if a customer's telephone numbers are stored in a table, and you want to have their home / work / mobile numbers in their own columns.

- Subqueries are very useful, but tying them to individual rows in your SELECT statement is a common performance trap. It forces SQL to recalculate the math for every single row. Instead, use subqueries to calculate your totals upfront and then join the results!

- Read from the inside out

- Defined and executed in the same statement.

- Defined inline within your main query.

- Only executed where defined and not reusable like a CTE.

Visit the [w3resource's SQLite Subqueries page](https://www.w3resource.com/sqlite/sqlite-subqueries.php) to learn more.



### EXAMPLE -- INNER JOIN WITH SUBQUERY

**Requirement**

From the `data` table, for each `city`, return the total counts or a distribution of tickets per `hour` on the biggest day (by events) by `createdDate`.

* Columns
  * `City`
  * `createdHour`
  * `countoccur`: the count of events

*    Exclude NULL cities i.e. `WHERE city IS NOT NULL`

* Sort
  *   `City` in ascending order
  *   `createdHour` in ascending order

**Pseudocode:**
*    Need to find the biggest day
*    JOIN to the `data` table
*    produce `SELECT` statement
*    include `WHERE` statement
*    `GROUP BY`
*    `ORDER BY`

In [38]:
def inner_join_example():

    # display the list of cities
    query_inner_join = '''
                SELECT d.city
                        ,subquery.createdymd
                        ,strftime('%H',CreatedDate) createdhour
                        ,count(*) countoccur
                FROM data d 

                -- SUBQUERY BELOW
                INNER JOIN ( --this select statement gets the date with the most events

                              SELECT strftime('%Y-%m-%d'
                                        ,CreatedDate) createdymd
                                        ,count(*) totalymd
                              FROM data
                              GROUP BY createdymd
                              ORDER BY totalymd desc
                              LIMIT 1
                            ) subquery
                        on strftime('%Y-%m-%d',d.CreatedDate) = subquery.createdymd
                -- END OF SUBQUERY

                WHERE d.city IS NOT NULL
                --sqlite is different than other SQL flavors in that you can reference aliases in the group by
                group by d.city, subquery.createdymd, createdhour
                order by d.city, subquery.createdymd, createdhour
                '''
    return query_inner_join

df_inner_join_example = pd.read_sql(inner_join_example(),conn_nyc)
display(df_inner_join_example)

,City,createdymd,createdhour,countoccur
0,ARVERNE,2014-11-18,11,1
1,ARVERNE,2014-11-18,12,2
2,ASTORIA,2014-11-18,03,1
3,ASTORIA,2014-11-18,07,1
4,ASTORIA,2014-11-18,10,2
...,...,...,...,...
623,Woodside,2014-11-18,13,1
624,Woodside,2014-11-18,15,1
625,Woodside,2014-11-18,16,1
626,Woodside,2014-11-18,17,1


**SIDE NOTE:**</br>
You can also reference column names by their position in the select statement, but that can be dangerous if someone else adds a column to your query at the beginning of the select statement. Also, it can be hard to read or troubleshoot if you have a 200 line SQL statement and don't remember the column positions. </br>
In a professional setting 0/10 - do not recommend, but if you're looking to save typing on an exam it's not going to break anything or if other instructors use the latter syntax and you are confused by it, the two are equivalent.

* GROUP BY d.city, subquery.createdymd, createdhour == GROUP BY 1,2,3
* ORDER BY d.city, subquery.createdymd, createdhour == ORDER BY 1,2,3


In [32]:
def verify_most_events_date():
    query_Most_Events = '''
                              SELECT strftime('%Y-%m-%d'
                                        ,CreatedDate) createdymd
                                        ,count(*) totalymd
                              FROM data
                              GROUP BY createdymd
                              ORDER BY totalymd desc

                            '''
    return query_Most_Events

df_verify_most_events_date = pd.read_sql(verify_most_events_date(),conn_nyc)
display(df_verify_most_events_date)

,createdymd,totalymd
0,2014-11-18,8466
1,2014-11-19,8333
2,2014-11-03,6966
3,2014-11-17,6424
4,2014-11-13,6424
5,2014-11-20,6412
6,2014-11-14,6300
7,2014-11-21,6273
8,2014-11-10,5757
9,2014-11-07,5456


In theory we should have found it's `2014-11-18` (8466 events)

### So what did we do here?

First the subquery itself:

1. The subquery counted the number of rows (complaints) for each date.
    ```sql
    SELECT strftime('%Y-%m-%d', CreatedDate) createdymd
         , count(*) totalymd
    FROM data
    GROUP BY createdymd
    ```

2. The subquery sorted by the number in descending order.
    ```sql
    SELECT strftime('%Y-%m-%d', CreatedDate) createdymd
         , count(*) totalymd
    FROM data
    GROUP BY createdymd
    ORDER BY totalymd desc
    LIMIT 1
    ```

3. The subquery returned one row, which is the date with the most complaints.

#### Next, in the main query:

1. The subquery inner joined to the main query on the date.
````sql         
                ...
                FROM data d 
                INNER JOIN (SELECT strftime('%Y-%m-%d'
                                        ,CreatedDate) createdymd
                                        ,count(*) totalymd
                              FROM data
                              GROUP BY createdymd
                              ORDER BY totalymd desc
                              LIMIT 1
                            ) subquery on strftime('%Y-%m-%d',d.CreatedDate) = subquery.createdymd
````

2. Because the join to the subquery is an inner join, it ensures that the only rows included/returned are those with that date.

The wrinkle with using limit 1 is what happens if there's two dates with a tie? We get whatever day the Query Engine decides is the top one. 
In this case, we know there's no ties, but in your day job or the exams (especially the hidden test cells in the exams), you can't assume that.
</br>
</br>
Below is an alternative way to accomplish the same query, but accounts for edge cases like ties, shows how you can use subqueries in the `WHERE` and `HAVING` clauses, and demonstrates subqueries that contain subqueries.

In [ ]:
def doublesubquery():

    # display the list of cities
    double_subquery = '''
                SELECT a.city
                        ,strftime('%Y-%m-%d',CreatedDate) createdymd
                        ,strftime('%H',CreatedDate) createdhour
                        ,count(*) countoccur
                FROM data a
                WHERE a.city IS NOT NULL
                and strftime('%Y-%m-%d',CreatedDate) in (
                    -- Subquery 1: Get the date(s) that have the MAX count
                        SELECT strftime('%Y-%m-%d', CreatedDate)
                        FROM data
                        GROUP BY strftime('%Y-%m-%d', CreatedDate)
                        HAVING count(*) = (
                            -- Subquery 2: Get the highest count value across all dates
                            SELECT MAX(daily_count)
                            FROM (
                                SELECT count(*) as daily_count
                                FROM data
                                GROUP BY strftime('%Y-%m-%d', CreatedDate)
                            )
                        )
                    )
                group by a.city, createdymd, createdhour
                order by a.city, createdymd, createdhour
                '''
    return double_subquery

df_doublesubquery_example = pd.read_sql(doublesubquery(),conn_nyc)
display(df_doublesubquery_example)

### LEFT JOIN WITH SUBQUERY AND MULTIPLE SUBQUERIES

From the `data` table, for each `city`, return the total counts or a distribution of tickets per `hour` on the biggest day (by events) by `createdDate`.

But this time, **only include `City` if they have an event on that day**.

Additionally, `City` and `Hour` combinations without events on that day should be included but should have a 0 for `countoccur`.

We know it's 2014-11-18 from before, but how do we put this into code dynamically?

* Columns
    * `City`
    * `createdHour`
    * `countoccur`: the count of events. **REMEMBER: This should be 0 if there aren't any events for that city/hour**

*    Exclude NULL cities i.e. `WHERE city IS NOT NULL`

* Sort
    * `City` in ascending order
    * `createdHour` in ascending order

**Pseudocode:**

*    Need to find the biggest day. This is query `MostEvents` below.

*    Need to get all the hours (0-23) that occur on the biggest day. This is query `DailyHours` below.

*    Need to get all the cities/hour combinations possible for the biggest day. This is query `Subquery1` below.

*    Need to get the counts for each city/hour on the biggest day, This is query `Subquery2` below.

*    Need to join query `Subquery1` with query `Subquery2` to retain all of query `Subquery1`.

*    produce `SELECT` statement

*    `GROUP BY`
*    `ORDER BY`

### Note that there are four distinct queries/subqueries:

MostEvents -  Gets the date with the most events. This subquery is written and executed in multiple places.

DailyHours -  Gets the distinct hours on the date with the most events. Depends on the date in `MostEvents`.

Subquery1 -  Gets a distinct list of city/createdhour for the date with the most events. Depends on the date in `MostEvents`.

Subquery2 -  Gets the number of events for city/hour. SAME AS INNER JOIN EXAMPLE ABOVE. Depends on the date in `MostEvents`.

In [ ]:
def left_join_example():

    # display the list of cities
    query_left_join = '''
                SELECT Subquery1.city
                        ,Subquery1.createdhour
                        ,COALESCE(Subquery2.countoccur,0) countoccur

                FROM ( --this gets a distinct list of city/createdhour for the date with the most events
                          SELECT DISTINCT a.city,DailyHours.createdhour
                          FROM data a
                          INNER JOIN
                                        ( --this join gets the date with the most events
                                                SELECT strftime('%Y-%m-%d',CreatedDate) createdymd
                                                        ,count(*) totalymd
                                                FROM data
                                                GROUP BY createdymd
                                                ORDER BY totalymd DESC
                                                LIMIT 1
                                        ) MostEvents on strftime('%Y-%m-%d',a.CreatedDate)=MostEvents.createdymd

                          LEFT JOIN
                                      ( --this join gets the distinct hours on the date with the most events
                                        SELECT distinct strftime('%Y-%m-%d',CreatedDate) createdymd
                                                        ,strftime('%H',CreatedDate) createdhour
                                        FROM data

                                      ) DailyHours on strftime('%Y-%m-%d',a.CreatedDate)=DailyHours.createdymd
                          WHERE a.city IS NOT NULL
                    ) Subquery1

                LEFT JOIN ( --this join gets the number of events for city/hour. SAME AS JOIN EXAMPLE ABOVE
                                  SELECT a.city
                                          ,strftime('%H',CreatedDate) createdhour
                                          ,count(*) countoccur
                                  FROM data a
                                  INNER JOIN ( --this join gets the date with the most events
                                                SELECT strftime('%Y-%m-%d',CreatedDate) createdymd
                                                        ,count(*) totalymd
                                                FROM data
                                                GROUP BY createdymd
                                                ORDER BY totalymd DESC
                                                LIMIT 1
                                              ) MostEvents on strftime('%Y-%m-%d',a.CreatedDate)=MostEvents.createdymd
                                  WHERE a.city IS NOT NULL
                                  GROUP BY a.city, createdhour
                               ) Subquery2 ON Subquery1.city = Subquery2.city AND Subquery1.createdhour = Subquery2.createdhour
                ORDER BY Subquery1.city, Subquery1.createdhour
                '''
    return query_left_join

df_left_join_example = pd.read_sql(left_join_example(),conn_nyc)
display(df_left_join_example)